In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")

train_df["occupied_date"] = pd.to_datetime(train_df["occupied_date"])
val_df["occupied_date"] = pd.to_datetime(val_df["occupied_date"])

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)

Training shape: (9248, 20)
Validation shape: (1989, 20)


In [2]:
TARGET = "occupied_room_nights"

FEATURES = [
    "hotel",
    "room_category",
    "day_of_week",
    "day",
    "month",
    "year",
    "quarter",
    "day_of_year",
    "is_weekend",
    "season",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7"
]

CATEGORICAL_FEATURES = [
    "hotel",
    "room_category",
    "season"
]

NUMERICAL_FEATURES = [
    col for col in FEATURES
    if col not in CATEGORICAL_FEATURES
]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]


In [3]:
unique_dates = np.array(
    sorted(train_df["occupied_date"].unique())
)

tscv = TimeSeriesSplit(n_splits=3)

date_cv = []

for fold, (train_date_idx, val_date_idx) in enumerate(
    tscv.split(unique_dates),
    start=1
):
    fold_train_dates = unique_dates[train_date_idx]
    fold_val_dates = unique_dates[val_date_idx]

    fold_train_idx = np.flatnonzero(
        train_df["occupied_date"].isin(fold_train_dates)
    )

    fold_val_idx = np.flatnonzero(
        train_df["occupied_date"].isin(fold_val_dates)
    )

    date_cv.append((fold_train_idx, fold_val_idx))

    print(f"Fold {fold}")
    print(
        "Train:",
        fold_train_dates.min(),
        "to",
        fold_train_dates.max()
    )
    print(
        "Validation:",
        fold_val_dates.min(),
        "to",
        fold_val_dates.max()
    )
    print()

Fold 1
Train: 2015-07-29 00:00:00 to 2015-12-11 00:00:00
Validation: 2015-12-12 00:00:00 to 2016-04-25 00:00:00

Fold 2
Train: 2015-07-29 00:00:00 to 2016-04-25 00:00:00
Validation: 2016-04-26 00:00:00 to 2016-09-08 00:00:00

Fold 3
Train: 2015-07-29 00:00:00 to 2016-09-08 00:00:00
Validation: 2016-09-09 00:00:00 to 2017-01-22 00:00:00



In [4]:
def create_preprocessor():

    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_FEATURES
            ),
            (
                "numerical",
                "passthrough",
                NUMERICAL_FEATURES
            )
        ]
    )

In [5]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_params = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__max_depth": [10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.7, 0.9, 1.0]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter=10,
    scoring="neg_mean_absolute_error",
    cv=date_cv,
    random_state=42,
    n_jobs=1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Best RF parameters:")
print(rf_search.best_params_)

print("Best CV MAE:")
print(-rf_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best RF parameters:
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_features': 0.9, 'model__max_depth': None}
Best CV MAE:
2.3223480568757844


In [6]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

xgb_params = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [3, 4, 5, 6],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_params,
    n_iter=10,
    scoring="neg_mean_absolute_error",
    cv=date_cv,
    random_state=42,
    n_jobs=1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

print("Best XGBoost parameters:")
print(xgb_search.best_params_)

print("Best CV MAE:")
print(-xgb_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best XGBoost parameters:
{'model__subsample': 0.7, 'model__n_estimators': 500, 'model__min_child_weight': 5, 'model__max_depth': 4, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.8}
Best CV MAE:
2.3535335938238906


In [7]:
def evaluate_model(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    total_actual = np.abs(y_true).sum()

    wape = (
        np.abs(y_true - y_pred).sum()
        / total_actual * 100
        if total_actual > 0
        else np.nan
    )

    return {
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3),
        "WAPE (%)": round(wape, 3)
    }

In [8]:
rf_best = rf_search.best_estimator_
xgb_best = xgb_search.best_estimator_

rf_pred = np.maximum(
    rf_best.predict(X_val),
    0
)

xgb_pred = np.maximum(
    xgb_best.predict(X_val),
    0
)

naive_pred = np.maximum(
    val_df["lag_1"].to_numpy(),
    0
)

results = pd.DataFrame([
    {
        "Model": "Naive Baseline",
        **evaluate_model(y_val, naive_pred)
    },
    {
        "Model": "Tuned Random Forest",
        **evaluate_model(y_val, rf_pred)
    },
    {
        "Model": "Tuned XGBoost",
        **evaluate_model(y_val, xgb_pred)
    }
])

results = results.sort_values("MAE")

results

,Model,MAE,RMSE,WAPE (%)
0,Naive Baseline,1.866,3.899,10.678
2,Tuned XGBoost,1.945,3.732,11.129
1,Tuned Random Forest,2.004,3.858,11.469


In [9]:
from pathlib import Path
import joblib

ARTIFACT_DIR = Path("../backend/artifacts")
REPORT_DIR = Path("../reports")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    rf_best,
    ARTIFACT_DIR / "random_forest_tuned.joblib"
)

joblib.dump(
    xgb_best,
    ARTIFACT_DIR / "xgboost_tuned.joblib"
)

results.to_csv(
    REPORT_DIR / "day5_tuned_model_comparison.csv",
    index=False
)

print("Models and comparison results saved.")

Models and comparison results saved.
